# ECON6083: Machine Learning in Economics
## Lecture 10 Exercise: Optimal Policy Learning, Text as Data, and Images as Data

**Coverage:** Lecture 10


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
import urllib.request
import zipfile
import os
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)


---

## Part 1: Optimal Policy Learning

We simulate data with heterogeneous treatment effects and learn optimal treatment policies under different constraints.


### Q1.1: Simulate Heterogeneous Treatment Effects

Generate a dataset with covariates $X$, treatment $W$, and outcome $Y$ where the true treatment effect varies by $X$.


In [2]:
n = 1000
X = np.random.normal(0, 1, (n, 3))
true_tau = 2 + 1.5 * X[:, 0] - 1.0 * X[:, 1]
e = 1 / (1 + np.exp(-0.3 * X[:, 0]))
W = np.random.binomial(1, e)
Y0 = 5 + 0.5 * X[:, 0] + 0.3 * X[:, 1] + np.random.normal(0, 1, n)
Y1 = Y0 + true_tau + np.random.normal(0, 0.5, n)
Y = W * Y1 + (1 - W) * Y0
policy_df = pd.DataFrame({'Y': Y, 'W': W, 'X0': X[:, 0], 'X1': X[:, 1], 'X2': X[:, 2], 'true_tau': true_tau})

# Estimate CATE using separate regressions for treated and control
treated = policy_df[policy_df['W'] == 1]
control = policy_df[policy_df['W'] == 0]
reg1 = LinearRegression().fit(treated[['X0', 'X1', 'X2']], treated['Y'])
reg0 = LinearRegression().fit(control[['X0', 'X1', 'X2']], control['Y'])
mu1_hat = reg1.predict(policy_df[['X0', 'X1', 'X2']])
mu0_hat = reg0.predict(policy_df[['X0', 'X1', 'X2']])
policy_df['cate_hat'] = mu1_hat - mu0_hat

print(f"True CATE range: [{true_tau.min():.2f}, {true_tau.max():.2f}]")
print(f"Estimated CATE range: [{policy_df['cate_hat'].min():.2f}, {policy_df['cate_hat'].max():.2f}]")


True CATE range: [-4.43, 9.97]
Estimated CATE range: [-4.59, 9.98]


### Q1.2: Simple Policy Rule

Implement the threshold policy: treat if estimated CATE > 0.


In [3]:
policy_df['policy_simple'] = (policy_df['cate_hat'] > 0).astype(int)
welfare_simple = policy_df[policy_df['policy_simple'] == 1]['true_tau'].mean()

print(f"Simple policy treats {policy_df['policy_simple'].sum()} / {n} individuals")
print(f"Average true welfare among treated: {welfare_simple:.3f}")


Simple policy treats 872 / 1000 individuals
Average true welfare among treated: 2.522


### Q1.3: Cost-Based Policy

Suppose treatment cost $c = 1.5$. Treat only if benefit exceeds cost.


In [4]:
cost = 1.5
policy_df['net_benefit'] = policy_df['cate_hat'] - cost
policy_df['policy_cost'] = (policy_df['net_benefit'] > 0).astype(int)
welfare_cost = policy_df[policy_df['policy_cost'] == 1]['true_tau'].mean()

print(f"Cost-based policy treats {policy_df['policy_cost'].sum()} / {n} individuals")
print(f"Average true welfare among treated: {welfare_cost:.3f}")


Cost-based policy treats 613 / 1000 individuals
Average true welfare among treated: 3.207


### Q1.4: Budget-Constrained Selection

With a budget to treat only 30% of the population, select the top 30% by estimated CATE.


In [5]:
budget = 0.30
n_treat = int(budget * n)
top_individuals = policy_df.nlargest(n_treat, 'cate_hat')
policy_df['policy_budget'] = 0
policy_df.loc[top_individuals.index, 'policy_budget'] = 1
welfare_budget = policy_df[policy_df['policy_budget'] == 1]['true_tau'].mean()

print(f"Budget policy treats {policy_df['policy_budget'].sum()} / {n} individuals")
print(f"Average true welfare among treated: {welfare_budget:.3f}")


Budget policy treats 300 / 1000 individuals
Average true welfare among treated: 4.192


### Q1.5: Doubly Robust Welfare Estimation

Compute doubly robust (DR) scores to improve welfare estimates.

$$\hat{\Gamma}_i = \hat{\tau}(X_i) + \frac{W_i (Y_i - \hat{\mu}_1(X_i))}{\hat{e}(X_i)} - \frac{(1-W_i)(Y_i - \hat{\mu}_0(X_i))}{1-\hat{e}(X_i)}$$


In [6]:
# Estimate propensity score
ps_model = LogisticRegression(max_iter=200).fit(policy_df[['X0', 'X1', 'X2']], policy_df['W'])
e_hat = np.clip(ps_model.predict_proba(policy_df[['X0', 'X1', 'X2']])[:, 1], 0.01, 0.99)

W_arr = policy_df['W'].values
Y_arr = policy_df['Y'].values
dr_scores = (policy_df['cate_hat'].values +
             W_arr * (Y_arr - mu1_hat) / e_hat -
             (1 - W_arr) * (Y_arr - mu0_hat) / (1 - e_hat))
policy_df['dr_score'] = dr_scores

print(f"DR score mean: {dr_scores.mean():.3f}")
print(f"DR score std:  {dr_scores.std():.3f}")


DR score mean: 1.997
DR score std:  2.781


### Q1.6: Policy Tree

Learn an interpretable policy rule by training a regression tree to predict **DR scores** from covariates.

**Why regression?** A standard decision tree classifier minimises classification error on a binary target (e.g. `dr_score > 0`), which is not the same as maximising policy welfare. A regression tree predicts the expected welfare gain for each individual; we then treat only those whose predicted gain exceeds the cost. This directly optimises for policy value.

*(In production, `econml.policy.PolicyTree` or `econml.policy.PolicyForest` would be preferred as they optimise the policy value directly.)*


In [7]:
X_features = policy_df[['X0', 'X1', 'X2']]

# Train regression tree to predict DR scores directly
policy_tree_reg = DecisionTreeRegressor(max_depth=3, random_state=42)
policy_tree_reg.fit(X_features, policy_df['dr_score'])

# Policy: treat if predicted DR score exceeds cost
predicted_dr = policy_tree_reg.predict(X_features)
policy_df['policy_tree'] = (predicted_dr > cost).astype(int)

welfare_tree = policy_df[policy_df['policy_tree'] == 1]['true_tau'].mean()

print(f"Policy tree treats {policy_df['policy_tree'].sum()} / {n} individuals")
print(f"Average true welfare among treated: {welfare_tree:.3f}")

from sklearn.tree import export_text
print("\nPolicy Tree Rules (predicted DR score at each leaf):")
print(export_text(policy_tree_reg, feature_names=['X0', 'X1', 'X2']))


Policy tree treats 531 / 1000 individuals
Average true welfare among treated: 3.309

Policy Tree Rules (predicted DR score at each leaf):
|--- X0 <= 0.38
|   |--- X1 <= -0.53
|   |   |--- X0 <= -0.42
|   |   |   |--- value: [1.66]
|   |   |--- X0 >  -0.42
|   |   |   |--- value: [3.12]
|   |--- X1 >  -0.53
|   |   |--- X0 <= -0.90
|   |   |   |--- value: [-1.00]
|   |   |--- X0 >  -0.90
|   |   |   |--- value: [1.01]
|--- X0 >  0.38
|   |--- X1 <= 0.55
|   |   |--- X0 <= 1.14
|   |   |   |--- value: [3.70]
|   |   |--- X0 >  1.14
|   |   |   |--- value: [5.28]
|   |--- X1 >  0.55
|   |   |--- X1 <= 1.39
|   |   |   |--- value: [2.57]
|   |   |--- X1 >  1.39
|   |   |   |--- value: [0.76]



### Q1.7: Policy Comparison

Compare all learned policies on true welfare.


In [8]:
results = pd.DataFrame({
    'Policy': ['Simple (CATE>0)', 'Cost-based (CATE>1.5)', 'Budget (top 30%)', 'Policy Tree'],
    'Treated': [policy_df['policy_simple'].sum(), policy_df['policy_cost'].sum(),
                policy_df['policy_budget'].sum(), policy_df['policy_tree'].sum()],
    'Avg_Welfare': [welfare_simple, welfare_cost, welfare_budget, welfare_tree]
})
print(results.to_string(index=False))


               Policy  Treated  Avg_Welfare
      Simple (CATE>0)      872     2.521550
Cost-based (CATE>1.5)      613     3.206906
     Budget (top 30%)      300     4.191936
          Policy Tree      531     3.308840


---

## Part 3: Text as Data

We analyse **real US Congressional floor-debate transcripts** from the Convote dataset (Thomas, Pang & Lee, EMNLP 2006). Each speech is labeled by whether the representative voted **Yea** (support) or **Nay** (oppose) on the bill being debated.

**Source:** Cornell University — `https://www.cs.cornell.edu/home/llee/data/convote.html`
**Dataset:** `exercises/data/congressional_speeches.csv` (3,052 speeches from the 109th Congress)


### What We Are Predicting in This Section

In Part 3, we use **real US Congressional floor speeches** from the 109th Congress (2005–2006) to tackle a concrete prediction task:

> **Given the text of a Representative's floor speech about a bill, can we predict whether they voted **Yea** (support) or **Nay** (oppose) on that bill?**

This mirrors a classic political-economy question: *Can we infer legislators' actions from their words?*  The dataset contains 3,052 speeches, each paired with the speaker's actual recorded vote.  We will extract text features (TF-IDF, EPU keywords, BERT embeddings, Word2Vec, SBERT) and train classifiers to see how well language alone predicts the vote.


### Q3.1: Load Congressional Speeches & TF-IDF

Load the Convote dataset of real congressional floor-debate speeches and vectorise them using TF-IDF.


In [9]:
# Load real congressional floor-debate speeches
speech_path = os.path.join('..', 'exercises', 'data', 'congressional_speeches.csv')
# if not os.path.exists(speech_path):
#     # Fallback: download Convote dataset and extract speeches
#     print('Local dataset not found. Downloading Convote dataset...')
import urllib.request
import tarfile
convote_url = 'https://www.cs.cornell.edu/home/llee/data/convote/convote_v1.1.tar.gz'
tar_path = os.path.join(data_dir, 'convote_v1.1.tar.gz')
urllib.request.urlretrieve(convote_url, tar_path)
with tarfile.open(tar_path, 'r:gz') as tar:
    tar.extractall(data_dir)
print('Convote downloaded. Please re-run this cell.')

speech_df = pd.read_csv(speech_path)
print(f'Loaded {len(speech_df)} real congressional speeches')
print(f'Vote distribution: {speech_df["vote"].value_counts().to_dict()}')
print('\nSample Yea speech:')
print(speech_df[speech_df['vote'] == 'Yea'].iloc[0]['text'][:300] + '...')
print('\nSample Nay speech:')
print(speech_df[speech_df['vote'] == 'Nay'].iloc[0]['text'][:300] + '...')

# TF-IDF vectorization on real congressional speeches
vectorizer = TfidfVectorizer(max_features=300, stop_words='english', ngram_range=(1, 2))
X_tfidf = vectorizer.fit_transform(speech_df['text'])
y_labels = speech_df['label'].values  # 1 = Yea, 0 = Nay
print(f'\nTF-IDF shape: {X_tfidf.shape}')
print(f'Top features: {list(vectorizer.get_feature_names_out()[:15])}')


NameError: name 'data_dir' is not defined

### Q3.2: Economic Policy Uncertainty (EPU) Keywords in Congressional Debate

Baker, Bloom & Davis construct the EPU index by counting newspaper articles containing uncertainty-related terms. We replicate this logic on **real congressional floor speeches** to see which debates mention economic uncertainty.


In [10]:
# EPU keywords from Baker, Bloom & Davis methodology
uncertainty_terms = ['uncertain', 'uncertainty', 'risk', 'risky', 'volatile', 'volatility',
                     'crisis', 'recession', 'downturn', 'turmoil', 'instability']
policy_terms = ['policy', 'regulation', 'regulatory', 'legislation', 'government',
                'federal reserve', 'fed', 'fiscal', 'monetary', 'tax', 'budget']
economy_terms = ['economy', 'economic', 'gdp', 'inflation', 'unemployment', 'markets',
                 'trade', 'deficit', 'spending', 'investment', 'jobs']

def count_epu_keywords(text, terms):
    text_lower = text.lower()
    return sum(1 for term in terms if term in text_lower)

speech_df['uncertainty_count'] = speech_df['text'].apply(
    lambda t: count_epu_keywords(t, uncertainty_terms))
speech_df['policy_count'] = speech_df['text'].apply(
    lambda t: count_epu_keywords(t, policy_terms))
speech_df['economy_count'] = speech_df['text'].apply(
    lambda t: count_epu_keywords(t, economy_terms))

# EPU score: at least one term from each category
speech_df['epu_speech'] = (
    (speech_df['uncertainty_count'] > 0) &
    (speech_df['policy_count'] > 0) &
    (speech_df['economy_count'] > 0)
).astype(int)

print(f'EPU-relevant speeches: {speech_df["epu_speech"].sum():,} / {len(speech_df):,} ({speech_df["epu_speech"].mean():.1%})')
print('\nSample EPU speeches:')
for _, row in speech_df[speech_df['epu_speech'] == 1].head(3).iterrows():
    print(f'  [{row["vote"]:3s}] {row["text"][:100]}...')

# Compare EPU mentions by vote direction
epu_by_vote = speech_df.groupby('vote')['epu_speech'].mean()
print('\nEPU mention rate by vote:')
print(f'  Yea speeches: {epu_by_vote["Yea"]:.1%}')
print(f'  Nay speeches: {epu_by_vote["Nay"]:.1%}')


NameError: name 'speech_df' is not defined

### Q3.3: Predicting Votes from Congressional Speech

Train a Naive Bayes classifier to predict whether a representative voted **Yea** or **Nay** based solely on the text of their floor speech. This mirrors how political scientists analyse legislative text to infer policy positions.


In [ ]:
# Split REAL congressional speech data
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y_labels, test_size=0.3, random_state=42, stratify=y_labels
)

clf = MultinomialNB()
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(f'Yea vs Nay Prediction Accuracy: {accuracy_score(y_test, y_pred):.1%}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Nay', 'Yea']))

# Most informative features for Yea vs Nay
feature_names = vectorizer.get_feature_names_out()
log_probs = clf.feature_log_prob_[1] - clf.feature_log_prob_[0]
top_yea = log_probs.argsort()[-10:][::-1]
top_nay = log_probs.argsort()[:10]
print('\nTop Yea-indicative words:')
for idx in top_yea:
    print(f'  {feature_names[idx]:20s}: {log_probs[idx]:.3f}')
print('\nTop Nay-indicative words:')
for idx in top_nay:
    print(f'  {feature_names[idx]:20s}: {log_probs[idx]:.3f}')


### Q3.4: BERT Embeddings for Vote Prediction

**BERT** revolutionised NLP by learning deep contextualised word representations from massive text corpora. Unlike TF-IDF, which treats words as independent tokens, BERT captures context and semantics.

We use **DistilBERT** (a lightweight BERT) to extract 768-dimensional embeddings from each congressional speech, then train a logistic regression classifier to predict Yea vs Nay.

**Requires:** `pip install transformers torch`


In [ ]:
import sys, subprocess

def install(*packages):
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--quiet',
        '--trusted-host', 'pypi.org',
        '--trusted-host', 'files.pythonhosted.org',
        *packages
    ])

# Auto-install transformers if missing
try:
    from transformers import DistilBertTokenizer, DistilBertModel
    import torch
    BERT_AVAILABLE = True
    print('transformers already installed')
except ImportError:
    print('Installing transformers...')
    install('transformers')
    from transformers import DistilBertTokenizer, DistilBertModel
    import torch
    BERT_AVAILABLE = True
    print('Installation complete.')


In [ ]:
import sys, subprocess

def install(*packages):
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--quiet',
        '--trusted-host', 'pypi.org',
        '--trusted-host', 'files.pythonhosted.org',
        *packages
    ])

# Auto-install transformers if missing
try:
    from transformers import DistilBertTokenizer, DistilBertModel
    import torch
    BERT_AVAILABLE = True
    print('transformers already installed')
except ImportError:
    print('Installing transformers...')
    install('transformers')
    from transformers import DistilBertTokenizer, DistilBertModel
    import torch
    BERT_AVAILABLE = True
    print('Installation complete.')


**Why does DistilBERT underperform TF-IDF here?**

| Reason | Explanation |
|---|---|
| **Small sample** | 800 speeches is tiny for BERT. Naive Bayes generalises better with limited data. |
| **No fine-tuning** | BERT knows Wikipedia/Books, not 2005 congressional jargon. Without domain adaptation it misses legislative cues. |
| **Loud keywords** | Partisan terms ('republican', 'small business', 'bush') are strong signals. TF-IDF catches them directly; frozen BERT spreads them across 768 dimensions. |
| **Overkill task** | Vote prediction here is mostly keyword detection, not deep semantic understanding. BERT's contextual power is wasted. |

> **When BERT would win:** 10K+ speeches with fine-tuning, longer nuanced texts where context matters, or cross-domain transfer.

This is why you should **always benchmark against a simple baseline**.


### Q3.5: Semantic Similarity Analysis with BERT

Beyond classification, BERT embeddings let us ask: *Do Yea and Nay speeches live in different semantic worlds?*
We compute cosine similarities between speech embeddings to measure how close representatives sound when they agree —
and whether some pairs sound remarkably similar even when they vote differently.


In [ ]:
if BERT_AVAILABLE:
    from sklearn.metrics.pairwise import cosine_similarity

    # Use bert_embeddings from Q3.4 (shape: n_speeches, 768)
    yea_mask = speech_df['vote'].head(len(bert_embeddings)) == 'Yea'
    nay_mask = ~yea_mask

    yea_embeds = bert_embeddings[yea_mask]
    nay_embeds = bert_embeddings[nay_mask]

    def avg_pairwise_cosine(embeds):
        """Average cosine similarity, excluding self-similarity."""
        sims = cosine_similarity(embeds)
        n = sims.shape[0]
        return (sims.sum() - n) / (n * (n - 1))

    yy_sim = avg_pairwise_cosine(yea_embeds)
    nn_sim = avg_pairwise_cosine(nay_embeds)
    yn_sim = cosine_similarity(yea_embeds, nay_embeds).mean()

    print('Average cosine similarity between speech embeddings:')
    print(f'  Yea ↔ Yea:   {yy_sim:.4f}')
    print(f'  Nay ↔ Nay:   {nn_sim:.4f}')
    print(f'  Yea ↔ Nay:   {yn_sim:.4f}')

    # Most similar Yea–Nay pair (opposite votes, closest language)
    cross_sims = cosine_similarity(yea_embeds, nay_embeds)
    max_pos = np.unravel_index(cross_sims.argmax(), cross_sims.shape)

    yea_indices = np.where(yea_mask)[0]
    nay_indices = np.where(nay_mask)[0]

    yea_idx = yea_indices[max_pos[0]]
    nay_idx = nay_indices[max_pos[1]]

    print(f"\nMost semantically similar Yea–Nay pair (sim={cross_sims[max_pos]:.4f}):")
    print(f"  [YEA] {speech_df.iloc[yea_idx]['text'][:250]}...")
    print(f"  [NAY] {speech_df.iloc[nay_idx]['text'][:250]}...")

    # Most dissimilar Yea–Yea pair (same vote, furthest language)
    yea_sims = cosine_similarity(yea_embeds)
    np.fill_diagonal(yea_sims, 1.0)          # exclude self
    min_yea_pos = np.unravel_index(yea_sims.argmin(), yea_sims.shape)

    print(f"\nMost semantically dissimilar Yea–Yea pair (sim={yea_sims[min_yea_pos]:.4f}):")
    print(f"  [YEA-1] {speech_df.iloc[yea_indices[min_yea_pos[0]]]['text'][:200]}...")
    print(f"  [YEA-2] {speech_df.iloc[yea_indices[min_yea_pos[1]]]['text'][:200]}...")
else:
    print('BERT not available — run Q3.4 first')


### Q3.6: Word Embeddings (Word2Vec)

Word2Vec trains a neural network to predict words from their neighbours, producing dense vectors where similar words cluster together. We train it on **real** congressional speeches.

**Requires:** `pip install gensim` (auto-installed below)


In [ ]:
import sys, subprocess

def install(*packages):
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--quiet',
        '--trusted-host', 'pypi.org',
        '--trusted-host', 'files.pythonhosted.org',
        *packages
    ])

try:
    from gensim.models import Word2Vec
    GENSIM_AVAILABLE = True
except ImportError:
    install('gensim')
    from gensim.models import Word2Vec
    GENSIM_AVAILABLE = True

if GENSIM_AVAILABLE:
    # Tokenise congressional speeches for Word2Vec
    tokenized_texts = [t.lower().split() for t in speech_df['text'].head(500)]

    w2v_model = Word2Vec(
        sentences=tokenized_texts,
        vector_size=100, window=5,
        min_count=2, workers=4,
        sg=1, seed=42
    )

    print(f'Trained Word2Vec on {len(tokenized_texts)} speeches')
    print(f'Vocabulary size: {len(w2v_model.wv)}')

    # Find most similar words to economic terms
    for word in ['economy', 'budget', 'tax', 'jobs']:
        if word in w2v_model.wv:
            similar = w2v_model.wv.most_similar(word, topn=5)
            print(f'\nMost similar to {word}:')
            for w, s in similar:
                print(f'  {w:15s}: {s:.3f}')
else:
    print('gensim not available — installation may have failed')


### Q3.7: Sentence Embeddings with SBERT

**SBERT** (Sentence-BERT) is specifically trained so that *cosine similarity between sentence embeddings directly measures semantic similarity*. This is more reliable than taking the BERT `[CLS]` token as a sentence proxy.

We extract SBERT embeddings for a sample of **real congressional speeches** and compute pairwise similarities.


In [ ]:
import sys, subprocess

def install(*packages):
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--quiet',
        '--trusted-host', 'pypi.org',
        '--trusted-host', 'files.pythonhosted.org',
        *packages
    ])

try:
    from sentence_transformers import SentenceTransformer
    SBERT_AVAILABLE = True
except ImportError:
    install('sentence-transformers')
    from sentence_transformers import SentenceTransformer
    SBERT_AVAILABLE = True

if SBERT_AVAILABLE:
    sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

    # Encode a sample of real congressional speeches
    sample_sbert = speech_df.head(100).copy()
    texts = sample_sbert['text'].tolist()
    sbert_embeddings = sbert_model.encode(texts, show_progress_bar=False)
    print(f'SBERT embedding shape: {sbert_embeddings.shape}')

    # Pairwise cosine similarity matrix (100 x 100)
    from sklearn.metrics.pairwise import cosine_similarity
    sims = cosine_similarity(sbert_embeddings)

    # Most similar pair overall
    # Exclude diagonal by setting it to -1
    np.fill_diagonal(sims, -1)
    i, j = np.unravel_index(sims.argmax(), sims.shape)
    print(f'\nMost semantically similar speech pair (cosine = {sims[i,j]:.3f}):')
    print(f"  [{sample_sbert.iloc[i]['vote']}] {sample_sbert.iloc[i]['text'][:120]}...")
    print(f"  [{sample_sbert.iloc[j]['vote']}] {sample_sbert.iloc[j]['text'][:120]}...")

    # Most similar Yea-Nay pair (opposite votes, closest language)
    yea_idx = sample_sbert[sample_sbert['vote'] == 'Yea'].index
    nay_idx = sample_sbert[sample_sbert['vote'] == 'Nay'].index
    yea_pos = [sample_sbert.index.get_loc(k) for k in yea_idx]
    nay_pos = [sample_sbert.index.get_loc(k) for k in nay_idx]

    if yea_pos and nay_pos:
        cross = sims[np.ix_(yea_pos, nay_pos)]
        yi, ni = np.unravel_index(cross.argmax(), cross.shape)
        yea_row = sample_sbert.iloc[yea_pos[yi]]
        nay_row = sample_sbert.iloc[nay_pos[ni]]
        print(f"\nMost similar Yea-Nay pair (cosine = {cross[yi,ni]:.3f}):")
        print(f"  [YEA] {yea_row['text'][:120]}...")
        print(f"  [NAY] {nay_row['text'][:120]}...")

    print("\nInterpretation: SBERT turns each speech into a 384-dim vector where")
    print("cosine similarity ≈ 1 means 'saying the same thing in different words'")
    print("and cosine similarity ≈ 0 means 'completely unrelated topics'.")
else:
    print('sentence-transformers not available')


---

## Part 4: Images as Data — Satellite Imagery and Economic Development

A growing body of research in development economics uses **satellite imagery** to measure economic activity where traditional survey data is scarce. Jean et al. (2016, *Science*) showed that CNN features from daytime satellite photos predict local poverty in Africa almost as well as expensive household surveys.

In this section we use **EuroSAT** — a real remote-sensing dataset of 27,000 satellite images labelled by land use — to replicate the core pipeline: download images, extract features, and predict economic development proxies.

**Dataset:** EuroSAT (Helber et al., 2019) — 64×64 pixel RGB satellite images
**Land-use classes:** AnnualCrop, Forest, HerbaceousVegetation, Highway, Industrial, Pasture, PermanentCrop, Residential, River, SeaLake
**Proxy:** Developed (Industrial / Highway / Residential) vs Undeveloped (Forest / Pasture / HerbaceousVegetation / AnnualCrop)

The cell below auto-installs required packages if they are missing.


In [ ]:
import sys, subprocess

def install(*packages):
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--quiet',
        '--trusted-host', 'pypi.org',
        '--trusted-host', 'files.pythonhosted.org',
        *packages
    ])

# Auto-install torch and torchvision if missing
try:
    import torch
    import torchvision
    print(f'torch {torch.__version__}, torchvision {torchvision.__version__} already installed')
except ImportError:
    print('Installing torch and torchvision...')
    install('torch', 'torchvision')
    import torch
    import torchvision
    print('Installation complete.')

import torchvision.transforms as transforms
TORCH_AVAILABLE = True

# Work around SSL certificate issues on some systems
import ssl
ssl._create_default_https_context = ssl._create_unverified_context


In [ ]:
if TORCH_AVAILABLE:
    # Download EuroSAT: real satellite imagery dataset
    eurosat = torchvision.datasets.EuroSAT(
        root='./lec10_data', download=True, transform=transforms.ToTensor()
    )

    print(f'Downloaded {len(eurosat):,} real satellite images')
    print(f'Classes: {eurosat.classes}')

    # Visualise one sample from each class
    class_samples = {}
    for img, label in eurosat:
        if label not in class_samples:
            class_samples[label] = img
        if len(class_samples) == len(eurosat.classes):
            break

    fig, axes = plt.subplots(2, 5, figsize=(14, 6))
    for idx, (label_idx, img) in enumerate(class_samples.items()):
        ax = axes.flat[idx]
        ax.imshow(img.permute(1, 2, 0))
        ax.set_title(eurosat.classes[label_idx], fontsize=10)
        ax.axis('off')
    plt.suptitle('EuroSAT Satellite Images by Land-Use Class')
    plt.tight_layout()
    plt.show()

    # Economic-development proxy labels
    developed = ['Industrial', 'Highway', 'Residential']
    undeveloped = ['Forest', 'Pasture', 'HerbaceousVegetation', 'AnnualCrop']
    dev_map = {}
    for i, c in enumerate(eurosat.classes):
        if c in developed:
            dev_map[i] = 1
        elif c in undeveloped:
            dev_map[i] = 0
        else:
            dev_map[i] = -1

    n_dev = sum(1 for l in dev_map.values() if l == 1)
    n_und = sum(1 for l in dev_map.values() if l == 0)
    print(f'\nProxy labels — Developed: {n_dev} classes, Undeveloped: {n_und} classes')
else:
    print('Install torch and torchvision to run this cell')


### Q4.2: Handcrafted Features for Land-Use Classification

Before deep learning, remote-sensing economists used **handcrafted spectral features** to classify land use. We compute simple but meaningful descriptors — brightness, greenness, contrast — and train a Random Forest to separate developed from undeveloped areas.


In [ ]:
if TORCH_AVAILABLE:
    # Extract handcrafted features from real satellite images
    n_sample = 3000
    # EuroSAT is ordered by class; shuffle to mix Developed and Undeveloped
    np.random.seed(42)
    shuffled_idx = np.random.permutation(len(eurosat))[:n_sample]
    features = []
    labels = []

    for idx in shuffled_idx:
        img, label = eurosat[idx]
        img_np = img.numpy()  # shape (3, 64, 64)
        r, g, b = img_np[0], img_np[1], img_np[2]

        # Remote-sensing style handcrafted features
        features.append([
            img_np.mean(),                    # overall brightness
            img_np.std(),                     # texture / contrast
            (g - r).mean(),                   # greenness index (veg vs built)
            (g / (r + 1e-6)).mean(),          # simple vegetation ratio
            ((r > g) & (r > b)).mean(),       # fraction of red-dominant pixels
        ])
        labels.append(dev_map[label])

    # Filter out neutral classes (River, SeaLake, PermanentCrop)
    X_sat = np.array([f for f, l in zip(features, labels) if l >= 0])
    y_sat = np.array([l for l in labels if l >= 0])

    # Train/test split
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_sat, y_sat, test_size=0.3, random_state=42, stratify=y_sat
    )

    from sklearn.ensemble import RandomForestClassifier
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_tr, y_tr)
    y_pred = rf.predict(X_te)

    acc = accuracy_score(y_te, y_pred)
    print(f'Developed vs Undeveloped classification (handcrafted features):')
    print(f'  Accuracy: {acc:.1%}')
    print(f'  N = {len(y_sat)} satellite images')

    feat_names = ['brightness', 'contrast', 'greenness', 'veg_ratio', 'red_dominant']
    print('\nFeature importances:')
    for name, imp in zip(feat_names, rf.feature_importances_):
        print(f'  {name:15s}: {imp:.3f}')
else:
    print('Install torch and torchvision to run this cell')


### Q4.3: Transfer Learning with ResNet-18 on Satellite Images

Modern remote-sensing research relies on **transfer learning**: a CNN pre-trained on millions of natural images (ImageNet) is repurposed to extract features from satellite photos. We use ResNet-18 to generate 512-dimensional feature vectors for each EuroSAT image, then train a logistic regression classifier — the same pipeline Jean et al. (2016) used for poverty prediction.


In [ ]:
if TORCH_AVAILABLE:
    # Load ResNet-18 pre-trained on ImageNet
    import ssl
    ssl._create_default_https_context = ssl._create_unverified_context
    resnet18 = torchvision.models.resnet18(weights='DEFAULT')
    cnn_extractor = torch.nn.Sequential(*list(resnet18.children())[:-1])
    cnn_extractor.eval()

    # ImageNet preprocessing (must match pre-training)
    preprocess = transforms.Compose([
        transforms.Resize(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    print('Extracting ResNet-18 CNN features from satellite images...')
    cnn_features = []
    cnn_labels = []

    n_cnn = min(5000, len(eurosat))
    # EuroSAT is ordered by class; shuffle to ensure both Developed and Undeveloped appear
    np.random.seed(42)
    shuffled_idx = np.random.permutation(len(eurosat))[:n_cnn]
    with torch.no_grad():
        for i, idx in enumerate(shuffled_idx):
            img_tensor = eurosat[idx][0]
            img_pil = transforms.ToPILImage()(img_tensor)
            img_tensor = preprocess(img_pil)
            feat = cnn_extractor(img_tensor.unsqueeze(0)).squeeze().numpy()
            label = dev_map[eurosat[idx][1]]
            if label >= 0:
                cnn_features.append(feat)
                cnn_labels.append(label)
            if (i + 1) % 1000 == 0:
                print(f'  Processed {i+1} / {len(shuffled_idx)} images')

    X_cnn = np.array(cnn_features)
    y_cnn = np.array(cnn_labels)
    print(f'\nCNN feature matrix: {X_cnn.shape}')

    # Logistic regression on CNN features
    Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(
        X_cnn, y_cnn, test_size=0.3, random_state=42, stratify=y_cnn
    )
    logreg = LogisticRegression(max_iter=500, C=1.0)
    logreg.fit(Xc_tr, yc_tr)
    cnn_acc = logreg.score(Xc_te, yc_te)

    print(f'\nResNet-18 + Logistic Regression on Satellite Images: {cnn_acc:.1%}')
    print('Compare: handcrafted spectral features (Q4.2) typically ~80-85%')
    print('Pre-trained CNNs capture spatial patterns (roads, roofs, fields)')
    print('that raw spectral statistics cannot.')
else:
    print('Install torch and torchvision to run this cell')
